<a href="https://colab.research.google.com/github/MujahidMalik7/Portfolio-/blob/main/AIKnowledge_Helper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
# List of required pip packages
pip_packages = [
    "sentence-transformers",
    "faiss-cpu",
    "pypdf",
    "pdf2image",
    "pytesseract",
    "transformers",
    "accelerate",
    "bitsandbytes",
    "langchain",
    "langchain-community",
    "langchain-text-splitters",
    "unstructured[all-docs]",
    "pillow"
]

# Function to check if a package is installed
import subprocess
import sys

def install_if_missing(packages):
    for package in packages:
        try:
            __import__(package.replace("-", "_").split("[")[0])  # Handle extras like [all-docs]
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        else:
            print(f"{package} is already installed.")

# Install only missing packages
install_if_missing(pip_packages)

# System dependencies (poppler + tesseract) - only install once per runtime
print("\nInstalling system dependencies (poppler-utils and tesseract-ocr)...")
!apt-get update -qq > /dev/null
!apt-get install -y tesseract-ocr poppler-utils -qq > /dev/null

print("\nAll required packages and system dependencies are ready!")

# Now import everything safely
import os
from pathlib import Path

from pypdf import PdfReader
from pdf2image import convert_from_path
import pytesseract

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline
import torch
from sentence_transformers import CrossEncoder

print("\nAll libraries imported successfully!")

sentence-transformers is already installed.
Installing faiss-cpu...
pypdf is already installed.
pdf2image is already installed.
pytesseract is already installed.
transformers is already installed.
accelerate is already installed.
bitsandbytes is already installed.
langchain is already installed.
langchain-community is already installed.
langchain-text-splitters is already installed.
unstructured[all-docs] is already installed.
Installing pillow...

Installing system dependencies (poppler-utils and tesseract-ocr)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

All required packages and system dependencies are ready!

All libraries imported successfully!


In [49]:
from unstructured.partition.auto import partition
from pdf2image import convert_from_path
import pytesseract
from PIL import Image
import os

def load_document(file_path: str) -> str:
    """
    Universal document loader using unstructured (auto-detects file type).
    Excellent for PDFs with tables, TXT, images, CSV, etc.
    """
    file_path = str(file_path)

    try:
        # Auto-detect file type and partition (handles PDF, TXT, DOCX, images, etc.)
        elements = partition(
            filename=file_path,
            strategy="hi_res",                  # Best for layout/table detection
            infer_table_structure=True,         # Extracts tables as clean text
            languages=["eng"],
            skip_infer_table_types=[]           # Force table inference for all
        )

        text_parts = []
        for element in elements:
            if element.category == "Table":
                text_parts.append(f"Table:\n{element.text}\n")
            else:
                text_parts.append(element.text)

        full_text = "\n\n".join([str(part) for part in text_parts if part])

        if len(full_text.strip()) > 50:
            print(f"Successfully extracted {os.path.basename(file_path)} with unstructured!")
            return full_text

    except Exception as e:
        print(f"Unstructured failed ({e}). Falling back to OCR...")

    # Reliable OCR fallback for any file (PDF or image)
    print("Running full OCR fallback...")
    if file_path.lower().endswith(".pdf"):
        images = convert_from_path(file_path, dpi=300)
    else:
        images = [Image.open(file_path)]

    text = ""
    for i, img in enumerate(images):
        print(f"OCR processing page/image {i+1}...")
        text += pytesseract.image_to_string(img) + "\n\n"

    return text

print("Universal document loader ready (great for PDFs with tables, TXT, images, etc.)!")

Universal document loader ready (great for PDFs with tables, TXT, images, etc.)!


In [107]:
embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

# ---- Text cleaner ----
def clean_text(text: str) -> str:
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = text.replace('\x0c', '')
    text = re.sub(r'\s+\n', '\n', text)
    return text.strip()

# ---- Text splitter ----
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""]
)

# ---- MAIN PIPELINE FUNCTION ----
def build_rag_pipeline(file_path: str):
    """
    Upload → load → clean → chunk → embed → FAISS index
    Returns everything needed for retrieval
    """
    print("\n📄 Loading document...")
    raw_text = load_document(file_path)

    if not raw_text.strip():
        raise ValueError("❌ Extracted text is empty")

    print("🧹 Cleaning text...")
    cleaned_text = clean_text(raw_text)

    print("✂️ Splitting into chunks...")
    chunks = text_splitter.split_text(cleaned_text)

    chunk_list = [{"id": i, "text": c.strip()} for i, c in enumerate(chunks)]
    print(f"✅ Created {len(chunk_list)} chunks")

    print("🔢 Creating embeddings...")
    embeddings = embedding_model.encode(
        [c["text"] for c in chunk_list],
        show_progress_bar=True,
        convert_to_numpy=True
    )

    faiss.normalize_L2(embeddings)

    print("📦 Building FAISS index...")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    print("🎯 RAG preprocessing complete!")

    return {
        "chunks": chunk_list,
        "index": index,
        "embeddings": embeddings
    }

In [99]:
import torch

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,   # float32 is fine and faster on CPU
    device_map="auto",          # Uses GPU if available, else CPU
    low_cpu_mem_usage=True
)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.1,
    do_sample=False
)

print(f"Loaded {model_name} (780M params) – great performance for lightweight RAG!")
print("Ready for high-quality answers.")

Device set to use cpu


Loaded google/flan-t5-large (780M params) – great performance for lightweight RAG!
Ready for high-quality answers.


In [109]:
# ========= CELL 5: UPLOAD + RUN PIPELINE =========

from google.colab import files

print("📤 Please upload your document (PDF / TXT / CSV / image):")
uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ No file uploaded")

file_path = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {file_path}")

# ---- Run full preprocessing pipeline ----
rag_data = build_rag_pipeline(file_path)

# Store globally for QA
chunk_list = rag_data["chunks"]
faiss_index = rag_data["index"]

print("\n🚀 RAG system is READY for questions!")


📤 Please upload your document (PDF / TXT / CSV / image):


Saving Mujahid_Resume.pdf to Mujahid_Resume.pdf

✅ Uploaded: Mujahid_Resume.pdf

📄 Loading document...
Successfully extracted Mujahid_Resume.pdf with unstructured!
🧹 Cleaning text...
✂️ Splitting into chunks...
✅ Created 4 chunks
🔢 Creating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📦 Building FAISS index...
🎯 RAG preprocessing complete!

🚀 RAG system is READY for questions!


In [117]:
from sentence_transformers import CrossEncoder

# Load reranker (do this once)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def retrieve_relevant_chunks(query: str, top_k: int = 10):
    query_emb = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)

    D, I = index.search(query_emb, top_k * 2)  # Get more candidates for reranking

    candidate_chunks = []
    scores = []
    seen = set()

    for score, idx in sorted(zip(D[0], I[0]), reverse=True):
        if idx not in seen and 0 <= idx < len(chunk_list):
            candidate_chunks.append(chunk_list[idx]["text"])
            scores.append(float(score))
            seen.add(idx)
        if len(candidate_chunks) >= top_k * 2:
            break

    return candidate_chunks, scores

def rerank_chunks(query: str, chunks: list) -> list:
    """Rerank using precise cross-encoder."""
    if len(chunks) <= 5:
        return chunks  # Skip if too few

    pairs = [[query, chunk] for chunk in chunks]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
    return [c for c, _ in ranked[:5]]  # Return top 5

def answer_question(query: str):
    # Retrieve more candidates
    candidate_chunks, faiss_scores = retrieve_relevant_chunks(query, top_k=10)

    # Rerank for precision
    final_chunks = rerank_chunks(query, candidate_chunks)
    context = "\n\n".join(final_chunks)

    # Overall relevance (from FAISS average)
    relevance = sum(faiss_scores[:len(final_chunks)]) / len(final_chunks) if faiss_scores else 0.0
    relevance = max(0.0, min(1.0, relevance))
    status = "✅ Excellent" if relevance > 0.7 else "✅ Good" if relevance > 0.55 else "⚠️ Moderate" if relevance > 0.4 else "⚠️ Low"
    print(f"Retrieval Relevance Score: {relevance:.3f}/1.00 {status}")

    # STRICT, GROUNDED PROMPT
    prompt = f"""You are a factual assistant. Answer the question using ONLY the context below.
- Quote exact phrases from the context when possible.
- Every claim must be directly supported by a sentence in the context.
- Do not infer, guess, or add external knowledge.
- If the answer is not explicitly stated, write 'It is not mentioned in the document'.
- Also try to understand the context of user if his query is a bit understandable and answer accordingly.

Context:
{context}

Question: {query}

Answer:"""

    result = generator(
        prompt,
        max_new_tokens=300,
        temperature=0.0,
        do_sample=False,
        repetition_penalty=1.3
    )

    answer = result[0]['generated_text'].strip()
    if answer.lower().startswith("answer:"):
        answer = answer[7:].strip()

    return answer

# === Clean Query ===
print("🚀 High-Accuracy RAG System Ready! (Cleaned + Reranked + Grounded)\n")
query = input("Ask a question about the document: ").strip()
if not query:
    query = "Summarize the document."

print("\n" + "="*60)
print("ANSWER:")
answer = answer_question(query)
print("\n" + answer)
print("="*60)

🚀 High-Accuracy RAG System Ready! (Cleaned + Reranked + Grounded)

Ask a question about the document: Which email is mentioned in the document?

ANSWER:
Retrieval Relevance Score: 0.396/1.00 ⚠️ Low

mujahidtufail726@gmail.com
